In [1]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

In [ ]:
# Lấy link cho từng sản phẩm ở trang danh sách sản phẩm
def get_product_link(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    box = soup.find("div", id="box_product")
    products = box.find_all("div", class_="col-sm-5ths")
    for product in products:
        link = product.find("a").get("href")
        product_link.append(link)

In [ ]:
# Lấy cấu hình chi tiết của từng sản phẩm
def get_detail(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, "html.parser")
    
        data= {}

        name = soup.find("h1")
        price = soup.find("span", class_="_price")
        data["Tên"] = name.text.strip() if name else None
        data["Giá"] = price.text.strip() if price else None

        table = soup.find("table", class_="charactestic_table_detail")
        if table:
            rows = table.find_all("tr")
            for row in rows:
                title = row.find("td", class_="title")
                content = row.find("td", class_="content_charactestic")
                if title and content:
                    data[title.text.strip()] = content.text.strip()
        return data
    except Exception as e:
        print(f"Lỗi tại {url}: {e}")
        return None

In [ ]:
product_link = []

page_link = [f"https://trungtran.vn/laptop/page-{i}/" for i in range(2, 58)]
page_link.insert(0, "https://trungtran.vn/laptop/")

# Chạy luồng song song để lấy link cho từng sản phẩm
with ThreadPoolExecutor(10) as exe:
    exe.map(get_product_link, page_link)

In [ ]:
# Chạy luồng song song để lấy cấu hình cho từng sản phẩm
with ThreadPoolExecutor(10) as exe:
    result = list(exe.map(get_detail, product_link))

df = pd.DataFrame(result)
df.to_csv("data/laptop.csv", index = False)